In [1]:
import pyphylon
import importlib
import os

from pyphylon.blast_utils import *
from pyphylon.util import load_config

In [2]:
CONFIG = load_config("config.yml")
SPECIES = CONFIG['PG_NAME']
temp_folder = CONFIG.get("REUSE_TEMP_DIR", "../temp/")
data_dir = CONFIG.get("SNAKEMAKE_DATA_DIR", "data/")

# Extract representative alleles
If you wish to compare the rerpesentative alleles for each gene cluster to external  sequences and annotations, you can extract them to a unique file using this command

In [3]:
# function to extract the representative alleles for each gene cluster
extract_reference_sequences(os.path.join(data_dir, 'processed/cd-hit-results'), SPECIES, os.path.join(data_dir, 'processed/cd-hit-results', f'{SPECIES}_representative_sequences'))

Skipped 516 representative sequences from filtered-out genomes


In [ ]:
extract_reference_dna_sequences(data_dir, SPECIES, os.path.join(data_dir, 'processed/cd-hit-results', f'{SPECIES}_representative_DNA_sequences'))

Skipped 516 representative sequences from filtered-out genomes


 24%|██▎       | 1916/8096 [00:48<02:22, 43.48it/s]

# Comparing pangenome against blast database of interest

Requires blast to be installed in your environment. If using conda, the command can be installed with `conda install -c bioconda blast`.

Example given for [VFDB](https://www.mgc.ac.cn/VFs/download.htm) for the download of the core dataset protein sequences.

In [ ]:
# Download of proteins fasta downloaded and placed into external directory in data outside of this notebook
# This can be done for any fasta of interest
make_blast_db(os.path.join(data_dir, 'external/VFDB/VFDB_setA_pro.fas'), os.path.join(data_dir, 'external/VFDB/VFDB')) # download VFDB database here: https://www.mgc.ac.cn/VFs/download.htm direct link: https://www.mgc.ac.cn/VFs/Down/VFDB_setA_pro.fas.gz)

In [ ]:
blast_localdb_enrichment(os.path.join(data_dir, 'external/VFDB/VFDB'), os.path.join(data_dir, 'processed/cd-hit-results', SPECIES + '_representative_sequences'),
                         os.path.join(data_dir, 'external/VFDB/results.txt'), e_val = 1e-5)

In [ ]:
blast_results = process_blast_results(os.path.join(data_dir, 'external/VFDB/results.txt'), e_val = 1e-5, percent_identity=80)
blast_results

# Make blast database from our pangenome

In [ ]:
DATABASE = os.path.join(data_dir, 'external/PangenomeDB/PangenomeDB')
INPUT_FILE = os.path.join(data_dir, 'processed/cd-hit-results', SPECIES)

make_blast_db(INPUT_FILE, DATABASE)

Create a query file of interest with a sequence you hope to blast, example sequence is from https://www.uniprot.org/uniprotkb/A0A4Y6ER29/entry#sequences


In [ ]:
QUERY_FILE = os.path.join(data_dir, 'external/PangenomeDB/query.fa')
OUTPUT_FILE = os.path.join(data_dir, 'external/PangenomeDB/results.txt')

blast_localdb_enrichment(DATABASE, QUERY_FILE, OUTPUT_FILE, e_val = 1e-5)

In [ ]:
blast_results = process_blast_results(OUTPUT_FILE, e_val = 1e-5, percent_identity=0, unique=False)
blast_results